# Occult CKD Detection Pipeline

Analysis code for:
> Ishida H, et al. **Interpretable Prediction Model for Occult CKD Detection**. *Kidney International Reports* (2026).

## Overview
This pipeline develops and validates an interpretable logistic regression model using six routine laboratory variables (Age, Sex, Cre, UN, Alb, Hb) to prioritize cystatin C testing for occult CKD detection.

**Occult CKD** is defined as eGFRcys < 60 mL/min/1.73 m² with eGFRcr ≥ 60 mL/min/1.73 m².

## Outputs
All outputs are CSV files corresponding to figures and tables in the manuscript:
- `Figure1_FlowCounts_occult.csv` — Study flow counts
- `Table1_Baseline_PrimaryCohort.csv` — Baseline characteristics
- `Figure2_ROC_External_occult.csv` — ROC curves
- `Table2_ExternalPerformance_occult.csv` — Performance metrics
- `Figure3_TopFrac_Subgroup_occult.csv` — Resource-constrained screening
- `Figure4_LogitCoefs_occult.csv` — Model coefficients
- `Figure5_DCA_occult.csv` — Decision curve analysis
- `Figure5_Calibration_occult.csv` — Calibration plots
- `SupplementaryTableS1_LogitParams.csv` — Implementation parameters
- `SupplementaryTableS2a/b_*_byCys60.csv` — Secondary analysis baseline
- `SupplementaryTableS3_ExternalPerformance_cys60.csv` — Secondary performance
- `SupplementaryFigureS1_ROC_External_cys60.csv` — Secondary ROC
- `SupplementaryFigureS2_TopFrac_Subgroup_cys60.csv` — Secondary screening


In [ ]:
import sys
sys.argv = sys.argv[:1]


In [ ]:
# ---- User settings ----
# Modify INPUT_CSV to point to your data file.
# TOTAL_LIS_N and EXCLUDED_CYSC_MISSING_N match the manuscript (Gifu University Hospital, 2022-2025).
INPUT_CSV = "total_eq5.csv"
OUTDIR = "out_final"
RUN_MODE = "both"
CUTOFF_DATE = "2025-01-01"
TOTAL_LIS_N = 273403
EXCLUDED_CYSC_MISSING_N = 266300


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Occult CKD detection pipeline.

Figure 5:
  Figure 5a (DCA): Proposed logistic regression / eGFReq5 / eGFRindiv / treat-all / treat-none
  Figure 5b (Calibration): Proposed / eGFReq5 / eGFRindiv

Inputs:
  total_eq5.csv (recommended) or any CSV that includes at least:
    Date, Age, Sex(1=Male,0=Female), Cre, CysC, UN, Alb, Hb
  Optional: HT, BW (for BMI); HT expected in cm if > 3 else meters.

Outputs (filenames fixed to manuscript plan):
  Main:
    Figure1_FlowCounts_occult.csv
    Table1_Baseline_PrimaryCohort.csv
    Figure2_ROC_External_occult.csv
    Table2_ExternalPerformance_occult.csv
    Figure3_TopFrac_Subgroup_occult.csv
    Figure4_LogitCoefs_occult.csv
    Figure5_DCA_occult.csv
    Figure5_Calibration_occult.csv
  Supplementary:
    SupplementaryTableS1_LogitParams.csv
    SupplementaryTableS2_Baseline_SecondaryCohort.csv
    SupplementaryTableS3_ExternalPerformance_cys60.csv
    SupplementaryFigureS1_ROC_External_cys60.csv
    SupplementaryFigureS2_TopFrac_Subgroup_cys60.csv

Author: Hidekazu Ishida (Gifu University Hospital)
"""
from __future__ import annotations

import argparse
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import json

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix
from sklearn.utils import resample

from scipy import stats

# -----------------------------
# Config / Labels
# -----------------------------
CUTOFF_DATE_DEFAULT = "2025-01-01"

LABEL_PROPOSED = "Proposed logistic regression"
LABEL_CRREF = "Cr-reference logistic regression"
LABEL_TREE = "Decision tree"
LABEL_HYBRID = "Hybrid stacking model"  # not used in final manuscript figures
LABEL_EGFRINDIV = "eGFRindiv"
LABEL_EGFREQ5 = "eGFReq5"
LABEL_EGFRINDIV_RULE = "eGFRindiv rule (<60)"
LABEL_EGFREQ5_RULE = "eGFReq5 rule (<60)"
LABEL_TREAT_ALL = "Treat-all"
LABEL_TREAT_NONE = "Treat-none"


def anthropometric_outlier_mask(df: pd.DataFrame) -> pd.Series:
    """Return boolean mask for anthropometric outliers using:
      Height <100 cm or >220 cm; Weight <30 kg or >200 kg.
    Height may be stored as cm (e.g., 165) or meters (e.g., 1.65).
    If HT/BW not present, returns all-False.
    """
    if ("HT" not in df.columns) or ("BW" not in df.columns):
        return pd.Series(False, index=df.index)
    ht_raw = pd.to_numeric(df["HT"], errors="coerce")
    bw = pd.to_numeric(df["BW"], errors="coerce")
    ht_cm = ht_raw.copy()
    ht_cm[ht_raw <= 3] = ht_raw[ht_raw <= 3] * 100.0
    outlier = (ht_cm.notna() & ((ht_cm < 100) | (ht_cm > 220))) | (bw.notna() & ((bw < 30) | (bw > 200)))
    outlier = outlier.fillna(False)
    return outlier

# -----------------------------
# Utility: eGFR formulas
# -----------------------------
def egfr_cr_jsn(cre_mgdl: np.ndarray, age: np.ndarray, sex01_male: np.ndarray) -> np.ndarray:
    """JSN creatinine-based eGFR (Japanese equation). Sex: 1=Male, 0=Female."""
    cre = np.asarray(cre_mgdl, dtype=float)
    age = np.asarray(age, dtype=float)
    male = np.asarray(sex01_male, dtype=int) == 1
    base = 194.0 * np.power(cre, -1.094) * np.power(age, -0.287)
    # female factor 0.739
    base[~male] *= 0.739
    return base

def egfr_cys_jsn(cys_mgL: np.ndarray, age: np.ndarray, sex01_male: np.ndarray) -> np.ndarray:
    """JSN cystatin C eGFR (Japanese equation): 104*CysC^-1.019*(0.996^Age) - 8; female *0.929."""
    cys = np.asarray(cys_mgL, dtype=float)
    age = np.asarray(age, dtype=float)
    male = np.asarray(sex01_male, dtype=int) == 1
    base = 104.0 * np.power(cys, -1.019) * np.power(0.996, age) - 8.0
    base[~male] *= 0.929
    return base

def egfr_eq5_ajkd2009(cre_mgdl: np.ndarray, age: np.ndarray, alb_gdl: np.ndarray, bun_mgdl: np.ndarray, sex01_male: np.ndarray) -> np.ndarray:
    """
    Japanese 5-variable eGFR equation (AJKD 2009; as provided by user):
      Eq5 = 142 * SCr^-0.923 * Age^-0.185 * Alb^0.414 * SUN^-0.233 * 0.772 (if female)
    Here: SCr=Cre, SUN=BUN(=UN).
    """
    cre = np.asarray(cre_mgdl, dtype=float)
    age = np.asarray(age, dtype=float)
    alb = np.asarray(alb_gdl, dtype=float)
    bun = np.asarray(bun_mgdl, dtype=float)
    male = np.asarray(sex01_male, dtype=int) == 1
    base = 142.0 * np.power(cre, -0.923) * np.power(age, -0.185) * np.power(alb, 0.414) * np.power(bun, -0.233)
    base[~male] *= 0.772
    return base

# -----------------------------
# Utility: metrics, CI, ROC
# -----------------------------
def safe_auc(y_true: np.ndarray, score: np.ndarray) -> float:
    y_true = np.asarray(y_true, dtype=int)
    score = np.asarray(score, dtype=float)
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(roc_auc_score(y_true, score))

def bootstrap_auc_ci(y_true: np.ndarray, score: np.ndarray,
                     n_boot: int = 1000, seed: int = 42, alpha: float = 0.05) -> Tuple[float, float, float]:
    """Bootstrap 95% CI for AUC."""
    rng = np.random.RandomState(seed)
    y_true = np.asarray(y_true, dtype=int)
    score = np.asarray(score, dtype=float)
    aucs = []
    n = len(y_true)
    for _ in range(n_boot):
        idx = rng.choice(n, n, replace=True)
        if len(np.unique(y_true[idx])) < 2:
            continue
        aucs.append(float(roc_auc_score(y_true[idx], score[idx])))
    if len(aucs) == 0:
        return (np.nan, np.nan, np.nan)
    lo = float(np.percentile(aucs, 100 * alpha / 2))
    hi = float(np.percentile(aucs, 100 * (1 - alpha / 2)))
    return (float(np.mean(aucs)), lo, hi)

def wilson_ci(k: int, n: int, alpha: float = 0.05) -> Tuple[float, float]:
    """Wilson score interval for a proportion."""
    if n <= 0:
        return (np.nan, np.nan)
    from math import sqrt
    # z for 1-alpha/2
    import scipy.stats as st
    z = float(st.norm.ppf(1 - alpha / 2))
    phat = k / n
    denom = 1 + z**2 / n
    center = (phat + z**2 / (2*n)) / denom
    half = (z * sqrt((phat*(1-phat) + z**2/(4*n)) / n)) / denom
    return (max(0.0, center - half), min(1.0, center + half))

def confusion_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    sens = tp / (tp+fn) if (tp+fn) > 0 else np.nan
    spec = tn / (tn+fp) if (tn+fp) > 0 else np.nan
    ppv  = tp / (tp+fp) if (tp+fp) > 0 else np.nan
    npv  = tn / (tn+fn) if (tn+fn) > 0 else np.nan
    acc  = (tp+tn) / (tp+tn+fp+fn) if (tp+tn+fp+fn) > 0 else np.nan
    return {
        "tp": tp, "fp": fp, "tn": tn, "fn": fn,
        "sensitivity": sens, "specificity": spec,
        "ppv": ppv, "npv": npv, "accuracy": acc
    }


def youden_threshold(y_true: np.ndarray, prob: np.ndarray) -> float:
    """Choose probability threshold that maximizes Youden's J (sensitivity + specificity - 1).
    If ties occur, choose the smallest threshold (more sensitive).
    """
    y = np.asarray(y_true, dtype=int)
    p = np.asarray(prob, dtype=float)
    p = np.clip(p, 1e-9, 1-1e-9)
    # candidate thresholds from unique probs
    thr = np.unique(p)
    if len(thr) > 2000:
        # downsample candidates for speed
        thr = np.quantile(p, np.linspace(0,1,2001))
        thr = np.unique(thr)
    best_j = -np.inf
    best_t = 0.5
    for t in thr:
        pred = (p >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0,1]).ravel()
        sens = tp/(tp+fn) if (tp+fn)>0 else 0.0
        spec = tn/(tn+fp) if (tn+fp)>0 else 0.0
        j = sens + spec - 1.0
        if (j > best_j) or (np.isclose(j, best_j) and t < best_t):
            best_j = j
            best_t = float(t)
    return best_t


def bootstrap_ci_sensitivity(y_true: np.ndarray, score: np.ndarray, frac: float, n_boot: int = 1000, seed: int = 42) -> Tuple[float, float, float]:
    """Top-frac flagged sensitivity with bootstrap 95% CI."""
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true, dtype=int)
    score = np.asarray(score, dtype=float)
    n = len(y_true)
    if n == 0:
        return (np.nan, np.nan, np.nan)
    k = max(1, int(round(n * frac)))
    order = np.argsort(score)[::-1]  # high score = high risk
    flagged = np.zeros(n, dtype=bool)
    flagged[order[:k]] = True
    sens = y_true[flagged].sum() / y_true.sum() if y_true.sum() > 0 else np.nan

    boots = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yt = y_true[idx]
        sc = score[idx]
        if yt.sum() == 0:
            continue
        kk = max(1, int(round(len(idx) * frac)))
        ord2 = np.argsort(sc)[::-1]
        flag2 = np.zeros(len(idx), dtype=bool)
        flag2[ord2[:kk]] = True
        boots.append(yt[flag2].sum() / yt.sum())
    if len(boots) < 20:
        return (sens, np.nan, np.nan)
    lo, hi = np.quantile(boots, [0.025, 0.975])
    return (sens, float(lo), float(hi))

# -----------------------------
# Modeling helpers
# -----------------------------
def build_models(seed: int = 42) -> Dict[str, object]:
    proposed = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=2000, solver="lbfgs"))
    ])
    crref = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=2000, solver="lbfgs"))
    ])
    tree = DecisionTreeClassifier(max_depth=4, random_state=seed)
    # Hybrid model not used in final analysis
    hybrid = None
    return {LABEL_PROPOSED: proposed, LABEL_CRREF: crref, LABEL_TREE: tree, LABEL_HYBRID: hybrid}

def fit_models(train_df: pd.DataFrame, mode: str, models: Dict[str, object]) -> Dict[str, object]:
    # Fit each model on its own complete-case design matrix to avoid length mismatch
    Xp, yp = design_matrix(train_df, mode, which="proposed")
    models[LABEL_PROPOSED].fit(Xp, yp.values.astype(int))

    Xc, yc = design_matrix(train_df, mode, which="crref")
    models[LABEL_CRREF].fit(Xc, yc.values.astype(int))

    # Tree uses proposed feature set
    models[LABEL_TREE].fit(Xp, yp.values.astype(int))

    return models

def predict_proba(models: Dict[str, object], df: pd.DataFrame, mode: str) -> Dict[str, np.ndarray]:
    Xp, _ = design_matrix(df, mode, which="proposed")
    Xc, _ = design_matrix(df, mode, which="crref")

    out = {}
    out[LABEL_PROPOSED] = models[LABEL_PROPOSED].predict_proba(Xp)[:,1]
    out[LABEL_CRREF] = models[LABEL_CRREF].predict_proba(Xc)[:,1]
    if models[LABEL_TREE] is not None:
        out[LABEL_TREE] = models[LABEL_TREE].predict_proba(Xp)[:,1]
    return out

def design_matrix(df: pd.DataFrame, mode: str, which: str = "proposed") -> Tuple[pd.DataFrame, pd.Series]:
    # target
    target = "y_occult" if mode == "occult" else "y_cys60"

    if which == "crref":
        cols = ["Age", "Sex", "Cre"]
    else:
        # implementable: Age, Sex, Cre, UN, Alb, Hb (BMI excluded for clinical implementability)
        cols = ["Age", "Sex", "Cre", "UN", "Alb", "Hb"]

    # Drop rows with missing/infinite in used predictors or target
    use_cols = cols + [target]
    tmp = df[use_cols].replace([np.inf, -np.inf], np.nan).dropna().copy()

    y = tmp[target].astype(int)
    X = tmp[cols].copy()
    return X, y

# -----------------------------
# Comparator probability mapping (for Figure 5 final)
# -----------------------------
@dataclass
class UnivLogitMap:
    intercept: float
    coef: float

    def predict_proba(self, score: np.ndarray) -> np.ndarray:
        z = self.intercept + self.coef * np.asarray(score, dtype=float)
        # stable sigmoid
        z = np.clip(z, -50, 50)
        return 1.0 / (1.0 + np.exp(-z))

def fit_univ_logit_mapping(train_y: np.ndarray, score: np.ndarray) -> UnivLogitMap:
    """Fit 1-variable logistic regression y ~ score (score already oriented: higher = higher risk)."""
    lr = LogisticRegression(max_iter=2000, solver="lbfgs")
    X = np.asarray(score, dtype=float).reshape(-1,1)
    y = np.asarray(train_y, dtype=int)
    lr.fit(X, y)
    return UnivLogitMap(intercept=float(lr.intercept_[0]), coef=float(lr.coef_[0,0]))

# -----------------------------
# Cohort construction & baseline
# -----------------------------
def compute_bmi(df: pd.DataFrame) -> pd.DataFrame:
    if ("HT" not in df.columns) or ("BW" not in df.columns):
        return df
    ht = pd.to_numeric(df["HT"], errors="coerce")
    bw = pd.to_numeric(df["BW"], errors="coerce")
    # if HT looks like cm (e.g., 160), convert to meters
    ht_m = ht.copy()
    ht_m[ht > 3] = ht[ht > 3] / 100.0
    bmi = bw / (ht_m**2)
    df["BMI"] = bmi
    return df

def clean_input(df: pd.DataFrame) -> pd.DataFrame:
    # ensure required cols
    required = ["Date","Age","Sex","Cre","CysC","UN","Alb","Hb"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")
    out = df.copy()
    out["Date"] = pd.to_datetime(out["Date"], errors="coerce")
    for c in ["Age","Sex","Cre","CysC","UN","Alb","Hb"]:
        out[c] = pd.to_numeric(out[c], errors="coerce")

    # --- Anthropometric outlier exclusion (if available) ---
    # Excluded:
    #   Height < 100 cm or > 220 cm
    #   Weight < 30 kg or > 200 kg
    # Height may be stored as cm (e.g., 165) or meters (e.g., 1.65)
    if ("HT" in out.columns) and ("BW" in out.columns):
        ht_raw = pd.to_numeric(out["HT"], errors="coerce")
        bw = pd.to_numeric(out["BW"], errors="coerce")
        ht_cm = ht_raw.copy()
        ht_cm[ht_raw <= 3] = ht_raw[ht_raw <= 3] * 100.0
        outlier = (ht_cm.notna() & ((ht_cm < 100) | (ht_cm > 220))) | (bw.notna() & ((bw < 30) | (bw > 200)))
        out = out.loc[~outlier].copy()

    # Optional BMI computation (not used as a model feature)
    out = compute_bmi(out)

    # Drop rows with missing required fields
    out = out.dropna(subset=["Date","Age","Sex","Cre","CysC","UN","Alb","Hb"]).copy()

    # Basic validity: equations require strictly positive values (avoid inf from power with 0 or negative values)
    out = out[
        (out["Age"] > 0) &
        (out["Cre"] > 0) &
        (out["CysC"] > 0) &
        (out["UN"] > 0) &
        (out["Alb"] > 0)
    ].copy()

    # Replace inf/-inf in numeric columns
    num_cols = out.select_dtypes(include=[np.number]).columns
    out[num_cols] = out[num_cols].replace([np.inf, -np.inf], np.nan)

    out = out.reset_index(drop=True)
    return out

def add_egfr_and_outcomes(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["eGFRcr"] = egfr_cr_jsn(out["Cre"].to_numpy(), out["Age"].to_numpy(), out["Sex"].to_numpy())
    out["eGFRcys"] = egfr_cys_jsn(out["CysC"].to_numpy(), out["Age"].to_numpy(), out["Sex"].to_numpy())
    out["Eq5"] = egfr_eq5_ajkd2009(out["Cre"].to_numpy(), out["Age"].to_numpy(), out["Alb"].to_numpy(), out["UN"].to_numpy(), out["Sex"].to_numpy())
    # outcomes
    out["y_cys60"] = (out["eGFRcys"] < 60).astype(int)
    out["y_occult"] = ((out["eGFRcys"] < 60) & (out["eGFRcr"] >= 60)).astype(int)
    return out

def temporal_split(df: pd.DataFrame, cutoff_date: str) -> Tuple[pd.DataFrame, pd.DataFrame]:
    cutoff = pd.to_datetime(cutoff_date)
    internal = df[df["Date"] < cutoff].copy()
    external = df[df["Date"] >= cutoff].copy()
    return internal, external

def baseline_table(df: pd.DataFrame) -> pd.DataFrame:
    """Simple baseline summary: N, mean±sd for continuous, count(%) for binary sex."""
    rows = []
    n = len(df)
    rows.append({"variable":"N","value":n})
    for c in ["Age","Cre","CysC","UN","Alb","Hb","eGFRcr","eGFRcys","Eq5"]:
        if c in df.columns:
            v = df[c].dropna().to_numpy(dtype=float)
            rows.append({"variable":c, "value":f"{np.mean(v):.2f} ± {np.std(v, ddof=1):.2f}" if len(v)>1 else f"{np.mean(v):.2f}"})
    if "BMI" in df.columns:
        v = df["BMI"].dropna().to_numpy(dtype=float)
        rows.append({"variable":"BMI", "value":f"{np.mean(v):.2f} ± {np.std(v, ddof=1):.2f}" if len(v)>1 else f"{np.mean(v):.2f}"})
    # Sex
    male = int((df["Sex"]==1).sum())
    female = int((df["Sex"]==0).sum())
    rows.append({"variable":"Male, n (%)", "value":f"{male} ({male/n*100:.1f}%)" if n else "NA"})
    rows.append({"variable":"Female, n (%)", "value":f"{female} ({female/n*100:.1f}%)" if n else "NA"})
    return pd.DataFrame(rows)


def _median_iqr_series(s: pd.Series) -> Tuple[str, str]:
    v = pd.to_numeric(s, errors="coerce").dropna().to_numpy(dtype=float)
    if len(v) == 0:
        return ("", "")
    med = float(np.median(v))
    q1 = float(np.quantile(v, 0.25))
    q3 = float(np.quantile(v, 0.75))
    # formatting: keep 1 decimal for large, 2-3 for small
    def fmt(x):
        return f"{x:.1f}" if abs(x) >= 10 else f"{x:.2f}"
    return (fmt(med), f"[{fmt(q1)}-{fmt(q3)}]")

def _p_mwu(a: pd.Series, b: pd.Series) -> str:
    a = pd.to_numeric(a, errors="coerce").dropna()
    b = pd.to_numeric(b, errors="coerce").dropna()
    if len(a) < 3 or len(b) < 3:
        return ""
    try:
        p = stats.mannwhitneyu(a, b, alternative="two-sided").pvalue
    except Exception:
        return ""
    return "<0.001" if p < 0.001 else f"{p:.3g}"

def _p_chi2(a_pos: int, a_n: int, b_pos: int, b_n: int) -> str:
    if a_n == 0 or b_n == 0:
        return ""
    tbl = np.array([[a_pos, a_n-a_pos],[b_pos, b_n-b_pos]])
    try:
        chi2, p, dof, exp = stats.chi2_contingency(tbl)
    except Exception:
        return ""
    return "<0.001" if p < 0.001 else f"{p:.3g}"

def secondary_baseline_by_cys60(df: pd.DataFrame, cohort_label: str) -> pd.DataFrame:
    """Supplementary Table S2 panel: baseline characteristics by eGFRcys<60 within a cohort (internal or external).
    Continuous: median [IQR], Mann–Whitney U. Categorical: n(%), chi-square.
    Uses the secondary analysis cohort definition (CysC-available entire population)."""
    # Define groups
    g0 = df[df["y_cys60"] == 0].copy()
    g1 = df[df["y_cys60"] == 1].copy()

    def _prep(d: pd.DataFrame) -> pd.DataFrame:
        dd = d.copy()
        # HT/BW standardization
        if "HT" in dd.columns:
            ht = pd.to_numeric(dd["HT"], errors="coerce")
            dd["HT_cm"] = ht.where(ht>3, ht*100.0)
        else:
            dd["HT_cm"] = np.nan
        if "BW" in dd.columns:
            dd["BW_kg"] = pd.to_numeric(dd["BW"], errors="coerce")
        else:
            dd["BW_kg"] = np.nan
        # BMI
        ht_m = dd["HT_cm"]/100.0
        dd["BMI"] = dd["BW_kg"] / (ht_m**2)
        # eGFRindiv
        dd["eGFRindiv"] = egfr_indiv_mlmin(dd["eGFRcr"], dd["HT_cm"], dd["BW_kg"]) if "eGFRcr" in dd.columns else np.nan
        return dd

    g0 = _prep(g0)
    g1 = _prep(g1)

    n0, n1 = len(g0), len(g1)

    rows = []
    # N
    rows.append({"Cohort": cohort_label, "Variable": "N",
                 "eGFRcys ≥60": f"{n0:,}", "eGFRcys ≥60_IQR": "",
                 "eGFRcys <60": f"{n1:,}", "eGFRcys <60_IQR": "",
                 "P": "-"})

    # Sex male
    m0 = int((g0["Sex"] == 1).sum()) if n0 else 0
    m1 = int((g1["Sex"] == 1).sum()) if n1 else 0
    psex = _p_chi2(m0, n0, m1, n1)
    rows.append({"Cohort": cohort_label, "Variable": "Male, n (%)",
                 "eGFRcys ≥60": f"{m0:,} ({(m0/n0*100):.1f}%)" if n0 else "",
                 "eGFRcys ≥60_IQR": "",
                 "eGFRcys <60": f"{m1:,} ({(m1/n1*100):.1f}%)" if n1 else "",
                 "eGFRcys <60_IQR": "",
                 "P": psex})

    # Continuous variables to include (pragmatic set)
    cont = [
        ("Age (years)", "Age"),
        ("HT (cm)", "HT_cm"),
        ("BW (kg)", "BW_kg"),
        ("BMI (kg/m²)", "BMI"),
        ("Cre (mg/dL)", "Cre"),
        ("UN (mg/dL)", "UN"),
        ("Alb (g/dL)", "Alb"),
        ("Hb (g/dL)", "Hb"),
        ("eGFRcr (mL/min/1.73m²)", "eGFRcr"),
        ("eGFRcys (mL/min/1.73m²)", "eGFRcys"),
        ("eGFReq5 (mL/min/1.73m²)", "Eq5"),
        ("eGFRindiv (mL/min)", "eGFRindiv"),
    ]

    for label, col in cont:
        if col not in g0.columns or col not in g1.columns:
            rows.append({"Cohort": cohort_label, "Variable": label,
                         "eGFRcys ≥60": "", "eGFRcys ≥60_IQR": "",
                         "eGFRcys <60": "", "eGFRcys <60_IQR": "",
                         "P": ""})
            continue
        v0, iqr0 = _median_iqr_series(g0[col])
        v1, iqr1 = _median_iqr_series(g1[col])
        p = _p_mwu(g0[col], g1[col])
        rows.append({"Cohort": cohort_label, "Variable": label,
                     "eGFRcys ≥60": v0, "eGFRcys ≥60_IQR": iqr0,
                     "eGFRcys <60": v1, "eGFRcys <60_IQR": iqr1,
                     "P": p})

    return pd.DataFrame(rows)



def body_surface_area_m2(ht_cm: np.ndarray, bw_kg: np.ndarray) -> np.ndarray:
    """Du Bois BSA: 0.007184 * HT^0.725 * BW^0.425 (HT in cm, BW in kg)."""
    ht = np.asarray(ht_cm, dtype=float)
    bw = np.asarray(bw_kg, dtype=float)
    return 0.007184 * np.power(ht, 0.725) * np.power(bw, 0.425)

def egfr_indiv_mlmin(egfrcr_mlmin173: np.ndarray, ht_cm: np.ndarray, bw_kg: np.ndarray) -> np.ndarray:
    """Individualized eGFR adjusted by BSA: eGFRcr * BSA / 1.73."""
    bsa = body_surface_area_m2(ht_cm, bw_kg)
    return np.asarray(egfrcr_mlmin173, dtype=float) * bsa / 1.73

def median_iqr(x: pd.Series) -> Tuple[str, str]:
    v = pd.to_numeric(x, errors="coerce").dropna().to_numpy(dtype=float)
    if len(v) == 0:
        return ("", "")
    med = np.median(v)
    q1 = np.quantile(v, 0.25)
    q3 = np.quantile(v, 0.75)
    return (f"{med:.3g}" if med>=10 else f"{med:.3f}", f"[{q1:.3g}-{q3:.3g}]")

def p_mannwhitney(a: pd.Series, b: pd.Series) -> str:
    a = pd.to_numeric(a, errors="coerce").dropna()
    b = pd.to_numeric(b, errors="coerce").dropna()
    if len(a) < 3 or len(b) < 3:
        return ""
    try:
        p = stats.mannwhitneyu(a, b, alternative="two-sided").pvalue
    except Exception:
        return ""
    if p < 0.001:
        return "<0.001"
    return f"{p:.3g}"

def p_chi2_2x2(male_a: int, n_a: int, male_b: int, n_b: int) -> str:
    if n_a == 0 or n_b == 0:
        return ""
    tbl = np.array([[male_a, n_a - male_a],[male_b, n_b - male_b]])
    try:
        chi2, p, dof, exp = stats.chi2_contingency(tbl)
    except Exception:
        return ""
    if p < 0.001:
        return "<0.001"
    return f"{p:.3g}"

def table1_baseline_occult(primary_internal: pd.DataFrame, primary_external: pd.DataFrame) -> pd.DataFrame:
    """Create Table 1 matching the PPTX structure: within each cohort compare Normal vs Occult CKD.
    Normal/Occult are defined within Cr-normal (eGFRcr>=60) population:
      - Occult CKD: y_occult=1
      - Normal: y_occult=0
    """
    def prep(df: pd.DataFrame) -> pd.DataFrame:
        d = df.copy()
        # HT/BW cm/kg expected; if meters, convert
        if "HT" in d.columns:
            ht = pd.to_numeric(d["HT"], errors="coerce")
            d["HT_cm"] = ht.where(ht>3, ht*100.0)
        else:
            d["HT_cm"] = np.nan
        if "BW" in d.columns:
            d["BW_kg"] = pd.to_numeric(d["BW"], errors="coerce")
        else:
            d["BW_kg"] = np.nan
        # BMI for descriptive (not a feature)
        if "BMI" not in d.columns and ("HT_cm" in d.columns) and ("BW_kg" in d.columns):
            ht_m = d["HT_cm"]/100.0
            d["BMI"] = d["BW_kg"] / (ht_m**2)
        # eGFRindiv for descriptive
        d["eGFRindiv"] = egfr_indiv_mlmin(d["eGFRcr"], d["HT_cm"], d["BW_kg"])
        # rename Eq5 -> eGFReq5
        d["eGFReq5"] = d["Eq5"] if "Eq5" in d.columns else np.nan
        return d

    I = prep(primary_internal)
    E = prep(primary_external)

    def build_rows(df: pd.DataFrame):
        normal = df[df["y_occult"]==0]
        occult = df[df["y_occult"]==1]
        nN, nO = len(normal), len(occult)
        rows = {}
        # N
        rows["N"] = (f"{nN:,}", "", f"{nO:,}", "", "-")
        # Male
        maleN = int((normal["Sex"]==1).sum())
        maleO = int((occult["Sex"]==1).sum())
        psex = p_chi2_2x2(maleN, nN, maleO, nO)
        rows["Male, n (%)"] = (f"{maleN:,}", f"({maleN/nN*100:.1f})" if nN else "", f"{maleO:,}", f"({maleO/nO*100:.1f})" if nO else "", psex)
        # continuous variables
        cont = [
            ("Age (years)", "Age"),
            ("HT (cm)", "HT_cm"),
            ("BW (kg)", "BW_kg"),
            ("BMI (kg/m²)", "BMI"),
            ("Cre (mg/dL)", "Cre"),
            ("UN (mg/dL)", "UN"),
            ("Alb (g/dL)", "Alb"),
            ("Hb (g/dL)", "Hb"),
            ("eGFRcr (mL/min/1.73m²)", "eGFRcr"),
            ("eGFRcys (mL/min/1.73m²)", "eGFRcys"),
            ("eGFReq5 (mL/min/1.73m²)", "eGFReq5"),
            ("eGFRindiv (mL/min)", "eGFRindiv"),
        ]
        for label, col in cont:
            vN, iqrN = median_iqr(normal[col])
            vO, iqrO = median_iqr(occult[col])
            p = p_mannwhitney(normal[col], occult[col])
            rows[label] = (vN, iqrN, vO, iqrO, p)
        return rows

    rowsI = build_rows(I)
    rowsE = build_rows(E)

    # Assemble wide table
    out_rows = []
    order = ["N","Male, n (%)","Age (years)","HT (cm)","BW (kg)","BMI (kg/m²)","Cre (mg/dL)","UN (mg/dL)","Alb (g/dL)","Hb (g/dL)",
             "eGFRcr (mL/min/1.73m²)","eGFRcys (mL/min/1.73m²)","eGFReq5 (mL/min/1.73m²)","eGFRindiv (mL/min)"]
    for var in order:
        i = rowsI.get(var, ("","","","",""))
        e = rowsE.get(var, ("","","","",""))
        out_rows.append({
            "Variable": var,
            "Internal_Normal": i[0], "Internal_Normal_IQR": i[1],
            "Internal_OccultCKD": i[2], "Internal_OccultCKD_IQR": i[3],
            "Internal_P": i[4],
            "External_Normal": e[0], "External_Normal_IQR": e[1],
            "External_OccultCKD": e[2], "External_OccultCKD_IQR": e[3],
            "External_P": e[4],
        })
    return pd.DataFrame(out_rows)


# -----------------------------
# Figure/Table outputs
# -----------------------------
def flow_counts(df_all: pd.DataFrame, mode: str, cutoff_date: str, anthro_excluded_n: int = 0) -> pd.DataFrame:
    """Counts for Figure 1 (simple, consistent). df_all is AFTER cleaning.
    If anthropometric exclusions were applied upstream, pass their count as anthro_excluded_n.
    """
    internal, external = temporal_split(df_all, cutoff_date)
    rows = [
        {"step":"Total records after cleaning", "n":len(df_all)},
    ]
    if anthro_excluded_n and anthro_excluded_n > 0:
        rows.append({"step":"Excluded: Anthropometric outliers", "n":int(anthro_excluded_n),
                     "details":"Height <100 cm or >220 cm; Weight <30 kg or >200 kg"})
    rows += [
        {"step":"Internal (train) period", "n":len(internal)},
        {"step":"External (test) period", "n":len(external)},
    ]
    target = "y_occult" if mode=="occult" else "y_cys60"
    if len(external):
        rows.append({"step":"External events", "n":int(external[target].sum())})
        rows.append({"step":"External non-events", "n":int((1-external[target]).sum())})
    return pd.DataFrame(rows)

def figure1_flowchart_counts(total_lis_n: int, excluded_cysc_missing_n: int, participants_cysc_n: int,
                             anthro_excluded_n: int, final_n: int,
                             cutoff_date: str,
                             impl_internal_n: int, impl_external_n: int,
                             primary_internal_n: int, primary_external_n: int,
                             sec_internal_n: int, sec_external_n: int) -> pd.DataFrame:
    """Create Figure 1 flowchart counts matching the manuscript figure."""
    rows = []
    rows.append({"box":"Data source / Study period", "n": int(total_lis_n)})
    rows.append({"box":"Excluded: Cystatin C measurement missing", "n": int(excluded_cysc_missing_n)})
    rows.append({"box":"Participants with Cystatin C available", "n": int(participants_cysc_n)})
    rows.append({"box":"Excluded: Anthropometric outliers", "n": int(anthro_excluded_n),
                 "criteria":"Height <100 cm or >220 cm; Weight <30 kg or >200 kg"})
    rows.append({"box":"Final Study Population", "n": int(final_n)})
    rows.append({"box":"Temporal Cohort Splitting", "n": np.nan, "cutoff_date": cutoff_date})
    rows.append({"box":"Internal Cohort (implementable)", "n": int(impl_internal_n), "period": f"Before {cutoff_date}"})
    rows.append({"box":"External Validation Cohort (implementable)", "n": int(impl_external_n), "period": f"From {cutoff_date}"})
    rows.append({"box":"PRIMARY Analysis (Occult CKD) Cr-normal: eGFRcr ≥60 (internal)", "n": int(primary_internal_n)})
    rows.append({"box":"PRIMARY Analysis (Occult CKD) Cr-normal: eGFRcr ≥60 (external)", "n": int(primary_external_n)})
    rows.append({"box":"SECONDARY Analysis (eGFRcys<60 overall) Entire population (internal)", "n": int(sec_internal_n)})
    rows.append({"box":"SECONDARY Analysis (eGFRcys<60 overall) Entire population (external)", "n": int(sec_external_n)})
    return pd.DataFrame(rows)

def roc_csv_external(ext_df: pd.DataFrame, mode: str, models: Dict[str, object], univ_maps: Dict[str, UnivLogitMap]) -> pd.DataFrame:
    """Figure 2 / Supp Fig S1: ROC curve points (FPR, TPR) for each model/comparator."""
    target = "y_occult" if mode=="occult" else "y_cys60"
    y = ext_df[target].astype(int).to_numpy()

    # model probs
    probs = predict_proba(models, ext_df, mode)

    # comparator scores -> probability mapping (for ROC either is fine; AUC invariant under monotone transform)
    score_indiv = -ext_df["eGFRcr"].to_numpy(dtype=float)
    score_eq5 = -ext_df["Eq5"].to_numpy(dtype=float)

    p_indiv = univ_maps[LABEL_EGFRINDIV].predict_proba(score_indiv)
    p_eq5 = univ_maps[LABEL_EGFREQ5].predict_proba(score_eq5)

    series = {
        LABEL_PROPOSED: probs[LABEL_PROPOSED],
        LABEL_CRREF: probs[LABEL_CRREF],
        LABEL_TREE: probs.get(LABEL_TREE, np.full(len(y), np.nan)),
        # Hybrid intentionally omitted from this lean build; add back if you re-enable stacking
        LABEL_EGFRINDIV: p_indiv,
        LABEL_EGFREQ5: p_eq5,
    }
    rows = []
    for name, p in series.items():
        if np.all(np.isnan(p)):  # skip missing
            continue
        fpr, tpr, thr = roc_curve(y, p)
        auc = safe_auc(y, p)
        for i in range(len(fpr)):
            rows.append({
                "mode": mode,
                "dataset": "external",
                "model": name,
                "fpr": float(fpr[i]),
                "tpr": float(tpr[i]),
                "threshold": float(thr[i]),
                "auc": auc,
            })
    return pd.DataFrame(rows)

def external_performance_table(internal_df: pd.DataFrame, ext_df: pd.DataFrame, mode: str, models: Dict[str, object], univ_maps: Dict[str, UnivLogitMap]) -> pd.DataFrame:
    """Table 2 / Supp Table S3: overall performance.
    Thresholding policy (option B requested by user):
      - For probability-producing models/comparators, choose threshold on INTERNAL cohort by maximizing Youden's J,
        then apply that threshold to EXTERNAL for confusion-matrix metrics.
      - For fixed-rule comparators, use the rule directly (eGFR<60).
    Always report external AUC (threshold-independent) for probabilistic predictors.
    """
    target = "y_occult" if mode=="occult" else "y_cys60"
    y_ext = ext_df[target].astype(int).to_numpy()

    # model probs
    probs_ext = predict_proba(models, ext_df, mode)
    probs_int = predict_proba(models, internal_df, mode)

    # comparator probabilized (for AUC and Youden thresholding)
    score_indiv_int = -internal_df["eGFRcr"].to_numpy(dtype=float)
    score_eq5_int   = -internal_df["Eq5"].to_numpy(dtype=float)
    p_indiv_int = univ_maps[LABEL_EGFRINDIV].predict_proba(score_indiv_int)
    p_eq5_int   = univ_maps[LABEL_EGFREQ5].predict_proba(score_eq5_int)

    score_indiv_ext = -ext_df["eGFRcr"].to_numpy(dtype=float)
    score_eq5_ext   = -ext_df["Eq5"].to_numpy(dtype=float)
    p_indiv_ext = univ_maps[LABEL_EGFRINDIV].predict_proba(score_indiv_ext)
    p_eq5_ext   = univ_maps[LABEL_EGFREQ5].predict_proba(score_eq5_ext)

    # Collect probabilistic predictors
    prob_sets = [
        (LABEL_PROPOSED, probs_int[LABEL_PROPOSED], probs_ext[LABEL_PROPOSED]),
        (LABEL_CRREF, probs_int[LABEL_CRREF], probs_ext[LABEL_CRREF]),
    ]
    if LABEL_TREE in probs_ext:
        prob_sets.append((LABEL_TREE, probs_int.get(LABEL_TREE, np.full(len(internal_df), np.nan)), probs_ext[LABEL_TREE]))

    prob_sets += [
        (LABEL_EGFRINDIV, p_indiv_int, p_indiv_ext),
        (LABEL_EGFREQ5, p_eq5_int, p_eq5_ext),
    ]

    rows = []
    # Probabilistic models: threshold by Youden on INTERNAL
    y_int = internal_df[target].astype(int).to_numpy()
    for name, p_int, p_ext in prob_sets:
        p_int = np.asarray(p_int, dtype=float)
        p_ext = np.asarray(p_ext, dtype=float)
        if len(p_int) != len(y_int) or len(p_ext) != len(y_ext):
            continue
        auc = safe_auc(y_ext, p_ext)
        _, auc_lo, auc_hi = bootstrap_auc_ci(y_ext, p_ext, n_boot=1000, seed=42)
        thr = youden_threshold(y_int, p_int)
        yhat = (p_ext >= thr).astype(int)
        mtx = confusion_metrics(y_ext, yhat)
        rows.append({
            "mode": mode, "dataset":"external", "model": name,
            "threshold_method": "Youden (internal)",
            "threshold_value": thr,
            "auc": auc,
            "auc_ci_low": auc_lo,
            "auc_ci_high": auc_hi,
            **mtx
        })

    # Fixed rule comparators (<60) - confusion only
    yhat_indiv_rule = (ext_df["eGFRcr"].to_numpy(dtype=float) < 60).astype(int)
    yhat_eq5_rule   = (ext_df["Eq5"].to_numpy(dtype=float) < 60).astype(int)
    for name, yhat in [(LABEL_EGFRINDIV_RULE, yhat_indiv_rule), (LABEL_EGFREQ5_RULE, yhat_eq5_rule)]:
        mtx = confusion_metrics(y_ext, yhat)
        rows.append({
            "mode": mode, "dataset":"external", "model": name,
            "threshold_method": "Fixed rule",
            "threshold_value": np.nan,
            "auc": np.nan,
            "auc_ci_low": np.nan,
            "auc_ci_high": np.nan,
            **mtx
        })

    return pd.DataFrame(rows)

def resource_constrained_csv(ext_df: pd.DataFrame, mode: str, models: Dict[str, object], univ_maps: Dict[str, UnivLogitMap],
                            fracs: List[float] = [0.10, 0.15, 0.20]) -> pd.DataFrame:
    """Figure 3 / Supp Fig S2: Top-fraction flagged sensitivity (overall + subgroups), dot/CI-ready."""
    target = "y_occult" if mode=="occult" else "y_cys60"
    y = ext_df[target].astype(int).to_numpy()

    probs = predict_proba(models, ext_df, mode)
    score_indiv = -ext_df["eGFRcr"].to_numpy(dtype=float)
    score_eq5 = -ext_df["Eq5"].to_numpy(dtype=float)
    p_indiv = univ_maps[LABEL_EGFRINDIV].predict_proba(score_indiv)
    p_eq5 = univ_maps[LABEL_EGFREQ5].predict_proba(score_eq5)

    series = {
        LABEL_PROPOSED: probs[LABEL_PROPOSED],
        LABEL_CRREF: probs[LABEL_CRREF],
        LABEL_TREE: probs.get(LABEL_TREE, np.full(len(y), np.nan)),
        LABEL_EGFRINDIV: p_indiv,
        LABEL_EGFREQ5: p_eq5,
    }

    # subgroup masks
    def mask_all(df): return np.ones(len(df), dtype=bool)
    masks = [("Overall", mask_all(ext_df))]
    masks.append(("Male", (ext_df["Sex"].to_numpy()==1)))
    masks.append(("Female", (ext_df["Sex"].to_numpy()==0)))
    masks.append(("Age <70", (ext_df["Age"].to_numpy()<70)))
    masks.append(("Age ≥70", (ext_df["Age"].to_numpy()>=70)))
    if "BMI" in ext_df.columns:
        masks.append(("BMI <22", (ext_df["BMI"].to_numpy()<22)))
        masks.append(("BMI ≥22", (ext_df["BMI"].to_numpy()>=22)))
    masks.append(("Alb <4.0", (ext_df["Alb"].to_numpy()<4.0)))
    masks.append(("Alb ≥4.0", (ext_df["Alb"].to_numpy()>=4.0)))

    rows = []
    for frac in fracs:
        for sg, msk in masks:
            yy = y[msk]
            if len(yy) < 20:
                continue
            for name, p in series.items():
                if np.all(np.isnan(p)):
                    continue
                pp = np.asarray(p)[msk]
                sens, lo, hi = bootstrap_ci_sensitivity(yy, pp, frac=frac, n_boot=500, seed=123)
                rows.append({
                    "mode": mode, "dataset":"external",
                    "top_frac": frac,
                    "subgroup": sg,
                    "model": name,
                    "sensitivity": sens,
                    "sens_ci_low": lo,
                    "sens_ci_high": hi,
                    "n": int(len(yy)),
                    "events": int(yy.sum()),
                })
    return pd.DataFrame(rows)

def standardized_coef_bootstrap(train_df: pd.DataFrame, mode: str, n_boot: int = 1000, seed: int = 42) -> pd.DataFrame:
    """Figure 4: standardized coefficients of proposed model with bootstrap CI."""
    rng = np.random.default_rng(seed)
    X, y = design_matrix(train_df, mode, which="proposed")
    cols = X.columns.tolist()

    # Fit once to get standardized coeffs (scaler inside pipeline)
    pipe = Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(max_iter=2000, solver="lbfgs"))])
    pipe.fit(X, y)
    coef = pipe.named_steps["clf"].coef_[0]
    intercept = float(pipe.named_steps["clf"].intercept_[0])

    boots = {c: [] for c in cols}
    for _ in range(n_boot):
        idx = rng.integers(0, len(train_df), size=len(train_df))
        Xb = X.iloc[idx].reset_index(drop=True)
        yb = y.iloc[idx].reset_index(drop=True)
        if len(np.unique(yb)) < 2:
            continue
        pb = Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(max_iter=2000, solver="lbfgs"))])
        pb.fit(Xb, yb)
        cb = pb.named_steps["clf"].coef_[0]
        for c, v in zip(cols, cb):
            boots[c].append(float(v))

    rows = []
    for c, v in zip(cols, coef):
        arr = np.array(boots[c], dtype=float)
        lo, hi = (np.nan, np.nan) if len(arr) < 50 else tuple(np.quantile(arr, [0.025, 0.975]))
        rows.append({
            "mode": mode, "model": LABEL_PROPOSED,
            "feature": c,
            "standardized_coefficient": float(v),
            "ci_low": float(lo) if not np.isnan(lo) else np.nan,
            "ci_high": float(hi) if not np.isnan(hi) else np.nan,
            "intercept": intercept,
            "n_train": int(len(train_df)),
        })
    return pd.DataFrame(rows)

def implementation_params_csv(train_df: pd.DataFrame, mode: str, models: Dict[str, object]) -> pd.DataFrame:
    """Supplementary Table S1: mean/sd/coef/intercept for implementable logistic regression."""
    X, y = design_matrix(train_df, mode, which="proposed")
    scaler = StandardScaler().fit(X)
    Xs = scaler.transform(X)
    lr = LogisticRegression(max_iter=2000, solver="lbfgs").fit(Xs, y)
    means = scaler.mean_
    sds = np.sqrt(scaler.var_)
    rows = []
    for i, col in enumerate(X.columns):
        rows.append({
            "mode": mode,
            "feature": col,
            "mean": float(means[i]),
            "sd": float(sds[i]),
            "coef_standardized": float(lr.coef_[0,i]),
        })
    rows.append({
        "mode": mode,
        "feature": "__intercept__",
        "mean": np.nan,
        "sd": np.nan,
        "coef_standardized": float(lr.intercept_[0]),
    })
    return pd.DataFrame(rows)

# -----------------------------
# Figure 5 final: DCA + Calibration with probabilized eGFR comparators
# -----------------------------
def decision_curve_net_benefit(y: np.ndarray, p: np.ndarray, thresholds: np.ndarray) -> np.ndarray:
    """Net benefit curve for probability predictions p over thresholds."""
    y = np.asarray(y, dtype=int)
    p = np.asarray(p, dtype=float)
    n = len(y)
    out = []
    prev = y.mean() if n else np.nan
    for pt in thresholds:
        if pt <= 0 or pt >= 1:
            out.append(np.nan); continue
        pred = (p >= pt).astype(int)
        tp = int(((pred==1) & (y==1)).sum())
        fp = int(((pred==1) & (y==0)).sum())
        nb = (tp/n) - (fp/n) * (pt/(1-pt))
        out.append(nb)
    return np.asarray(out, dtype=float)

def decision_curve_csv(ext_df: pd.DataFrame, mode: str, models: Dict[str, object], univ_maps: Dict[str, UnivLogitMap],
                       thresholds: Optional[np.ndarray] = None) -> pd.DataFrame:
    """Figure 5a: DCA with Proposed + eGFReq5 + eGFRindiv + treat-all/none. (NO Hybrid)"""
    if thresholds is None:
        thresholds = np.round(np.arange(0.01, 0.51, 0.01), 2)
    target = "y_occult" if mode=="occult" else "y_cys60"
    y = ext_df[target].astype(int).to_numpy()
    n = len(y)
    prev = float(y.mean()) if n else np.nan

    probs = predict_proba(models, ext_df, mode)
    p_proposed = np.clip(probs[LABEL_PROPOSED], 1e-9, 1-1e-9)

    # probabilized comparators (fit on internal, passed in univ_maps)
    score_indiv = -ext_df["eGFRcr"].to_numpy(dtype=float)
    score_eq5 = -ext_df["Eq5"].to_numpy(dtype=float)
    p_indiv = np.clip(univ_maps[LABEL_EGFRINDIV].predict_proba(score_indiv), 1e-9, 1-1e-9)
    p_eq5 = np.clip(univ_maps[LABEL_EGFREQ5].predict_proba(score_eq5), 1e-9, 1-1e-9)

    nb_prop = decision_curve_net_benefit(y, p_proposed, thresholds)
    nb_ind  = decision_curve_net_benefit(y, p_indiv, thresholds)
    nb_eq5  = decision_curve_net_benefit(y, p_eq5, thresholds)

    # treat-all / treat-none
    nb_all = []
    nb_none = np.zeros_like(thresholds, dtype=float)
    for pt in thresholds:
        if pt <= 0 or pt >= 1:
            nb_all.append(np.nan); continue
        nb_all.append(prev - (1-prev) * (pt/(1-pt)))
    nb_all = np.asarray(nb_all, dtype=float)

    df = pd.DataFrame({
        "threshold": thresholds,
        "net_benefit_proposed_logistic": nb_prop,
        "net_benefit_eGFReq5": nb_eq5,
        "net_benefit_eGFRindiv": nb_ind,
        "net_benefit_treat_all": nb_all,
        "net_benefit_treat_none": nb_none,
        "n_external": n,
        "prevalence": prev,
    })
    return df

def calibration_curve_bins(y: np.ndarray, p: np.ndarray, n_bins: int = 10) -> pd.DataFrame:
    """Quantile bins calibration table with Wilson CI for observed rate."""
    y = np.asarray(y, dtype=int)
    p = np.asarray(p, dtype=float)
    n = len(y)
    if n == 0:
        return pd.DataFrame()
    # quantile bins
    qs = np.linspace(0, 1, n_bins+1)
    edges = np.quantile(p, qs)
    # make edges strictly increasing
    edges = np.unique(edges)
    if len(edges) < 3:
        # fallback: uniform bins
        edges = np.linspace(p.min(), p.max(), n_bins+1)
    # assign bins
    bin_idx = np.digitize(p, edges[1:-1], right=True)
    rows = []
    for b in np.unique(bin_idx):
        m = (bin_idx == b)
        yy = y[m]
        pp = p[m]
        if len(yy) == 0:
            continue
        obs = int(yy.sum())
        nn = int(len(yy))
        rate = obs/nn
        lo, hi = wilson_ci(obs, nn, alpha=0.05)
        rows.append({
            "bin_index": int(b),
            "n": nn,
            "mean_predicted": float(np.mean(pp)),
            "observed_event_rate": float(rate),
            "obs_ci_low": float(lo),
            "obs_ci_high": float(hi),
        })
    return pd.DataFrame(rows).sort_values("bin_index")

def calibration_slope_intercept(y: np.ndarray, p: np.ndarray) -> Tuple[float, float]:
    """Fit logistic regression of y on logit(p) to get calibration intercept and slope."""
    y = np.asarray(y, dtype=int)
    p = np.asarray(p, dtype=float)
    p = np.clip(p, 1e-9, 1-1e-9)
    logit = np.log(p/(1-p)).reshape(-1,1)
    if len(np.unique(y)) < 2:
        return (np.nan, np.nan)
    lr = LogisticRegression(max_iter=2000, solver="lbfgs").fit(logit, y)
    return (float(lr.intercept_[0]), float(lr.coef_[0,0]))

def calibration_csv(ext_df: pd.DataFrame, mode: str, models: Dict[str, object], univ_maps: Dict[str, UnivLogitMap], n_bins: int = 10) -> pd.DataFrame:
    """Figure 5b: Calibration for Proposed + eGFReq5 + eGFRindiv (probabilized)."""
    target = "y_occult" if mode=="occult" else "y_cys60"
    y = ext_df[target].astype(int).to_numpy()

    probs = predict_proba(models, ext_df, mode)
    p_proposed = np.clip(probs[LABEL_PROPOSED], 1e-9, 1-1e-9)

    score_indiv = -ext_df["eGFRcr"].to_numpy(dtype=float)
    score_eq5 = -ext_df["Eq5"].to_numpy(dtype=float)
    p_indiv = np.clip(univ_maps[LABEL_EGFRINDIV].predict_proba(score_indiv), 1e-9, 1-1e-9)
    p_eq5 = np.clip(univ_maps[LABEL_EGFREQ5].predict_proba(score_eq5), 1e-9, 1-1e-9)

    series = {
        LABEL_PROPOSED: p_proposed,
        LABEL_EGFREQ5: p_eq5,
        LABEL_EGFRINDIV: p_indiv,
    }

    out_rows = []
    for name, p in series.items():
        inter, slope = calibration_slope_intercept(y, p)
        bins = calibration_curve_bins(y, p, n_bins=n_bins)
        if bins.empty:
            continue
        bins = bins.copy()
        bins["mode"] = mode
        bins["dataset"] = "external"
        bins["model"] = name
        bins["calibration_intercept"] = inter
        bins["calibration_slope"] = slope
        out_rows.append(bins)
    return pd.concat(out_rows, ignore_index=True) if out_rows else pd.DataFrame()

# -----------------------------
# Orchestration
# -----------------------------

def filter_complete_cases(df: pd.DataFrame, mode: str) -> pd.DataFrame:
    """Filter to rows that are complete for ALL predictors needed for modeling in the given mode.

    IMPORTANT: preserves ALL columns (including derived eGFRcr/Eq5) by filtering rows, not subsetting columns.
    BMI is intentionally NOT required (and not used as a model feature) for clinical implementability.
    """
    target = "y_occult" if mode == "occult" else "y_cys60"
    cols = ["Age","Sex","Cre","UN","Alb","Hb", target]
    tmp = df[cols].replace([np.inf, -np.inf], np.nan).dropna()
    return df.loc[tmp.index].copy()

def build_univ_maps(train_df: pd.DataFrame, mode: str) -> Dict[str, UnivLogitMap]:
    """Fit internal mapping for comparators used in Figure 5 and ROC."""
    target = "y_occult" if mode=="occult" else "y_cys60"
    y = train_df[target].astype(int).to_numpy()

    score_indiv = -train_df["eGFRcr"].to_numpy(dtype=float)
    score_eq5 = -train_df["Eq5"].to_numpy(dtype=float)

    m_indiv = fit_univ_logit_mapping(y, score_indiv)
    m_eq5 = fit_univ_logit_mapping(y, score_eq5)
    return {LABEL_EGFRINDIV: m_indiv, LABEL_EGFREQ5: m_eq5}

def save_mode_outputs(mode: str, df_all: pd.DataFrame, cutoff_date: str, outdir: Path, anthro_excluded_n: int = 0, total_lis_n: int = 0, excluded_cysc_missing_n: int = 0, participants_cysc_n: int = 0, secondary_df: Optional[pd.DataFrame] = None) -> None:
    internal, external = temporal_split(df_all, cutoff_date)

    # complete-case filtering for consistent modeling
    internal = filter_complete_cases(internal, mode)
    external = filter_complete_cases(external, mode)

    # Primary analysis cohort (occult): Cr-normal eGFRcr ≥60 within implementable cohorts
    primary_internal = internal.loc[internal['eGFRcr'] >= 60].copy() if 'eGFRcr' in internal.columns else internal.copy()
    primary_external = external.loc[external['eGFRcr'] >= 60].copy() if 'eGFRcr' in external.columns else external.copy()

    # Secondary analysis (entire population with CysC available): use secondary_df if provided
    if secondary_df is not None:
        sec_internal, sec_external = temporal_split(secondary_df, cutoff_date)
        sec_internal_n = int(len(sec_internal))
        sec_external_n = int(len(sec_external))
    else:
        sec_internal_n = int(len(internal))
        sec_external_n = int(len(external))


    # complete-case filtering for consistent modeling
    internal = filter_complete_cases(internal, mode)
    external = filter_complete_cases(external, mode)

    # Fit models and comparator mappings on internal only
    models = build_models()
    models = fit_models(primary_internal if mode=='occult' else internal, mode, models)
    univ_maps = build_univ_maps(primary_internal if mode=='occult' else internal, mode)

    # Baselines
    if mode == "occult":
        figure1_flowchart_counts(total_lis_n=total_lis_n, excluded_cysc_missing_n=excluded_cysc_missing_n, participants_cysc_n=participants_cysc_n,
                             anthro_excluded_n=anthro_excluded_n, final_n=len(df_all), cutoff_date=cutoff_date,
                             impl_internal_n=len(internal), impl_external_n=len(external),
                             primary_internal_n=len(primary_internal), primary_external_n=len(primary_external),
                             sec_internal_n=sec_internal_n, sec_external_n=sec_external_n).to_csv(outdir / "Figure1_FlowCounts_occult.csv", index=False)
        table1_baseline_occult(primary_internal, primary_external).to_csv(outdir / "Table1_Baseline_PrimaryCohort.csv", index=False)
        roc_csv_external(primary_external if mode=="occult" else external, mode, models, univ_maps).to_csv(outdir / "Figure2_ROC_External_occult.csv", index=False)
        external_performance_table(primary_internal if mode=="occult" else internal, primary_external if mode=="occult" else external, mode, models, univ_maps).to_csv(outdir / "Table2_ExternalPerformance_occult.csv", index=False)
        resource_constrained_csv(primary_external if mode=="occult" else external, mode, models, univ_maps).to_csv(outdir / "Figure3_TopFrac_Subgroup_occult.csv", index=False)
        standardized_coef_bootstrap(primary_internal if mode=="occult" else internal, mode, n_boot=500).to_csv(outdir / "Figure4_LogitCoefs_occult.csv", index=False)

        # Figure 5 FINAL (review-safe): Proposed + eGFReq5 + eGFRindiv + treat-all/none; calibration 3 models
        decision_curve_csv(primary_external if mode=="occult" else external, mode, models, univ_maps).to_csv(outdir / "Figure5_DCA_occult.csv", index=False)
        calibration_csv(primary_external if mode=="occult" else external, mode, models, univ_maps, n_bins=10).to_csv(outdir / "Figure5_Calibration_occult.csv", index=False)

        implementation_params_csv(primary_internal if mode=="occult" else internal, mode, models).to_csv(outdir / "SupplementaryTableS1_LogitParams.csv", index=False)

    elif mode == "cys60":
        # supplementary-only items + required baseline secondary cohort        roc_csv_external(primary_external if mode=="occult" else external, mode, models, univ_maps).to_csv(outdir / "SupplementaryFigureS1_ROC_External_cys60.csv", index=False)
        # Supplementary Table S2 (B): stratified by eGFRcys<60 within internal and external cohorts
        if secondary_df is not None:
            sec_internal, sec_external = temporal_split(secondary_df, cutoff_date)
            secondary_baseline_by_cys60(sec_internal, cohort_label='Internal').to_csv(outdir / 'SupplementaryTableS2a_Internal_byCys60.csv', index=False)
            secondary_baseline_by_cys60(sec_external, cohort_label='External').to_csv(outdir / 'SupplementaryTableS2b_External_byCys60.csv', index=False)

        external_performance_table(internal, external, mode, models, univ_maps).to_csv(outdir / "SupplementaryTableS3_ExternalPerformance_cys60.csv", index=False)
        resource_constrained_csv(primary_external if mode=="occult" else external, mode, models, univ_maps).to_csv(outdir / "SupplementaryFigureS2_TopFrac_Subgroup_cys60.csv", index=False)

    # metadata
    meta = {
        "mode": mode,
        "cutoff_date": cutoff_date,
        "n_total": int(len(df_all)),
        "n_internal": int(len(internal)),
        "n_external": int(len(external)),
        "anthro_excluded_n": int(anthro_excluded_n),
    }
    (outdir / f"runmeta_{mode}.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")

def parse_args(argv: Optional[List[str]] = None) -> argparse.Namespace:
    p = argparse.ArgumentParser(add_help=True)
    p.add_argument("--input", type=str, default="total_eq5.csv", help="Input CSV path (e.g., total_eq5.csv)")
    p.add_argument("--outdir", type=str, default="outputs_ckd_pipeline", help="Output directory (will be created)")
    p.add_argument("--cutoff_date", type=str, default=CUTOFF_DATE_DEFAULT, help="Temporal split cutoff date (YYYY-MM-DD)")
    p.add_argument("--run_mode", type=str, default="both", choices=["occult","cys60","both"], help="Which endpoint(s) to run")
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--total_lis_n", type=int, default=None, help="Optional: total LIS records (for Figure 1 first box)")
    p.add_argument("--excluded_cysc_missing_n", type=int, default=None, help="Optional: number excluded due to missing CysC (for Figure 1)")
    # Important for Jupyter: ignore unknown args like -f
    ns, _unknown = p.parse_known_args(argv)
    return ns

def main(argv: Optional[List[str]] = None) -> None:
    ns = parse_args(argv)
    inpath = Path(ns.input)
    outdir = Path(ns.outdir)
    outdir.mkdir(parents=True, exist_ok=True)

    # --- Raw load (for Figure 1 upstream counts) ---
    df_raw = pd.read_csv(inpath)

    # Total LIS records (optional override to match manuscript)
    total_lis_n = int(ns.total_lis_n) if ns.total_lis_n is not None else int(len(df_raw))

    # Missing CysC count (optional override to match manuscript)
    if ns.excluded_cysc_missing_n is not None:
        excluded_cysc_missing_n = int(ns.excluded_cysc_missing_n)
    else:
        if "CysC" in df_raw.columns:
            excluded_cysc_missing_n = int(pd.to_numeric(df_raw["CysC"], errors="coerce").isna().sum())
        else:
            excluded_cysc_missing_n = 0

    # Participants with CysC available (per figure)
    participants_cysc_n = int(total_lis_n - excluded_cysc_missing_n)

    # Anthropometric exclusions should be counted among those with CysC available
    if "CysC" in df_raw.columns:
        mask_cysc = pd.to_numeric(df_raw["CysC"], errors="coerce").notna()
        anthro_excluded_n = int(anthropometric_outlier_mask(df_raw.loc[mask_cysc]).sum())
    else:
        anthro_excluded_n = int(anthropometric_outlier_mask(df_raw).sum())

    # --- Build analysis datasets ---
    # (1) Cleaned dataset for IMPLEMENTABLE model (requires UN/Alb/Hb etc.) — used for occult analyses + model fitting
    df_impl = clean_input(df_raw)  # applies anthropometric exclusion if HT/BW exist
    df_impl = add_egfr_and_outcomes(df_impl)

    # (2) Cleaned dataset for SECONDARY analysis "entire population" (requires CysC + minimal variables only)
    #     This matches the flowchart where secondary analysis counts differ from implementable cohort.
    df_min = df_raw.copy()
    # Required for secondary: Date, Age, Sex, Cre, CysC (entire population with CysC available)
    for c in ["Date","Age","Sex","Cre","CysC"]:
        if c not in df_min.columns:
            raise ValueError(f"Missing required column for secondary cohort: {c}")
    df_min["Date"] = pd.to_datetime(df_min["Date"], errors="coerce")
    for c in ["Age","Sex","Cre","CysC"]:
        df_min[c] = pd.to_numeric(df_min[c], errors="coerce")

    # Keep optional descriptive labs if present (to match Table 1 rows)
    for c in ["UN","Alb","Hb","HT","BW"]:
        if c in df_min.columns:
            df_min[c] = pd.to_numeric(df_min[c], errors="coerce")

    # keep only CysC available + core fields available
    df_min = df_min.dropna(subset=["Date","Age","Sex","Cre","CysC"]).copy()

    # apply anthropometric exclusion if HT/BW available
    if ("HT" in df_min.columns) and ("BW" in df_min.columns):
        ht_raw = pd.to_numeric(df_min["HT"], errors="coerce")
        bw = pd.to_numeric(df_min["BW"], errors="coerce")
        ht_cm = ht_raw.copy()
        ht_cm[ht_raw <= 3] = ht_raw[ht_raw <= 3] * 100.0
        outlier = (ht_cm.notna() & ((ht_cm < 100) | (ht_cm > 220))) | (bw.notna() & ((bw < 30) | (bw > 200)))
        df_min = df_min.loc[~outlier].copy()

    # add eGFR/outcomes
    df_min["eGFRcr"] = egfr_cr_jsn(df_min["Cre"].to_numpy(), df_min["Age"].to_numpy(), df_min["Sex"].to_numpy())
    df_min["eGFRcys"] = egfr_cys_jsn(df_min["CysC"].to_numpy(), df_min["Age"].to_numpy(), df_min["Sex"].to_numpy())

    # Eq5 (requires UN and Alb)
    if ("UN" in df_min.columns) and ("Alb" in df_min.columns):
        df_min["Eq5"] = egfr_eq5_ajkd2009(df_min["Cre"].to_numpy(), df_min["Age"].to_numpy(),
                                          df_min["Alb"].to_numpy(), df_min["UN"].to_numpy(), df_min["Sex"].to_numpy())
    else:
        df_min["Eq5"] = np.nan

    df_min["y_cys60"] = (df_min["eGFRcys"] < 60).astype(int)
    df_min["y_occult"] = ((df_min["eGFRcys"] < 60) & (df_min["eGFRcr"] >= 60)).astype(int)

    # Save outputs
    if ns.run_mode in ["occult","both"]:
        save_mode_outputs(
            "occult",
            df_impl,
            ns.cutoff_date,
            outdir,
            anthro_excluded_n=anthro_excluded_n,
            total_lis_n=total_lis_n,
            excluded_cysc_missing_n=excluded_cysc_missing_n,
            participants_cysc_n=participants_cysc_n,
            secondary_df=df_min,
        )
    if ns.run_mode in ["cys60","both"]:
        save_mode_outputs(
            "cys60",
            df_impl,
            ns.cutoff_date,
            outdir,
            anthro_excluded_n=anthro_excluded_n,
            total_lis_n=total_lis_n,
            excluded_cysc_missing_n=excluded_cysc_missing_n,
            participants_cysc_n=participants_cysc_n,
            secondary_df=df_min,
        )

    print(f"Completed. Outputs written to: {outdir.resolve()}")


In [ ]:
argv = [
    "--input", INPUT_CSV,
    "--outdir", OUTDIR,
    "--run_mode", RUN_MODE,
    "--cutoff_date", CUTOFF_DATE,
    "--total_lis_n", str(TOTAL_LIS_N),
    "--excluded_cysc_missing_n", str(EXCLUDED_CYSC_MISSING_N),
]
main(argv)


## Pipeline Summary

### 1. Data processing
- Reads patient data from input CSV (`total_eq5.csv`)
- Applies anthropometric outlier exclusion (height <100 cm or >220 cm; weight <30 kg or >200 kg)
- Computes eGFR values using Japanese Society of Nephrology equations

### 2. Outcome definitions
- **Occult CKD** (primary): eGFRcys < 60 with eGFRcr ≥ 60 mL/min/1.73 m²
- **cys60** (secondary): eGFRcys < 60 mL/min/1.73 m² regardless of eGFRcr

### 3. Models
- **Proposed**: L2-regularized logistic regression with StandardScaler (Age, Sex, Cre, UN, Alb, Hb)
- **Cr-reference**: Logistic regression with Age, Sex, Cre only
- **Decision tree**: Max depth 4, same 6-variable feature set
- **Comparators**: eGFReq5 and eGFRindiv (rule-based and probabilized)

### 4. Evaluation
- AUC with 1,000-iteration bootstrap 95% CI
- Threshold selection by Youden's J on internal cohort, applied to external
- Resource-constrained screening (top 10%, 15%, 20% flagged)
- Decision curve analysis and quantile calibration plots

### 5. Temporal split
- Development cohort: before 2025-01-01
- Validation cohort: from 2025-01-01 onward
